# **Typed tools**

Tools tipadas são ferramentas com um **contrato explícito para os dados que recebem** e, opcionalmente, para os dados que retornam. 

O contrato norteia o LLM e permite validação dos argumentos antes de executar a função.

- **Contrato da tool**: especifica argumentos aceitos e suas restrições.

**Exemplo:**

A tool `search_pubmed`, oriunda da função de mesmo nome, tem um contrato que estabelece:

- `query`: string obrigatória; 
- `limit`: inteiro entre 1 e 20; 
- **campos adicionais são proibidos!**.


Neste notebook, a construção das tools tipadas seguirá a boa conduta discutida no artigo: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents?utm_source=chatgpt.com

## **Import and configs**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

In [3]:
import json

from bibliometric_rag_assistant.ingestion.pubmed import search_pubmed
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY, PUBMED_SAMPLE_SIZE

## **Build the tool with Pydantic**

Usamos o Pydantic para centralizar a definir e validar os argumentos aceitos (a classe constrói o contrato!):

In [4]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class PubMedSearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    query: str = Field(
        min_length=1,
        description=(
            "PubMed query using Boolean operators and field tags, "
            "such as [Title/Abstract] and [MeSH Terms]."
        ),
    )

    limit: int = Field(
        ge=1,
        le=10_000,
        description="Maximum number of PMIDs requested for this search.",
    )
    

- `extra="forbid"`: rejeitar campos inesperados.
- `strict=True`: exigir os tipos definidos, evitando conversões como "5" para 5.
- `min_length=1`: rejeitar uma query vazia.
- `ge=1 e le=20`: limitar a quantidade solicitada.

O método `model_json_schema()` gera a descrição do contrato:

In [5]:
search_pubmed_tool = {
    "type": "function",
    "name": "search_pubmed",
    "description": (
        "Search PubMed and return the total number of matching records "
        "and a limited list of PMIDs. "
        "Use this to discover records; obtain publication details "
        "with a separate metadata tool."
    ),
    "parameters": PubMedSearchArgs.model_json_schema(),
    "strict": True,
}

In [6]:
search_pubmed_tool["parameters"]

{'additionalProperties': False,
 'properties': {'query': {'description': 'PubMed query using Boolean operators and field tags, such as [Title/Abstract] and [MeSH Terms].',
   'minLength': 1,
   'title': 'Query',
   'type': 'string'},
  'limit': {'description': 'Maximum number of PMIDs requested for this search.',
   'maximum': 10000,
   'minimum': 1,
   'title': 'Limit',
   'type': 'integer'}},
 'required': ['query', 'limit'],
 'title': 'PubMedSearchArgs',
 'type': 'object'}

O método `.model_validate_json()` valida dados passados nos argumentos e **cria uma instância**. Nesta validação o `args` é criado de maneira artificial, sem chamadas ao LLM:

In [16]:
args = PubMedSearchArgs.model_validate_json(
    '{"query": "PMIDS", "limit": 1}'
)


type(args), args.query, args.limit, args.model_dump()

(__main__.PubMedSearchArgs, 'PMIDS', 1, {'query': 'PMIDS', 'limit': 1})

In [8]:
args = PubMedSearchArgs.model_validate_json(
    '{"query": "", "limit": 5}'
)

type(args), args.query, args.limit, args.model_dump()

ValidationError: 1 validation error for PubMedSearchArgs
query
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short

## **Connect the Pydantic contract to a real tool execution**

- **Boa conduta: devolver ao modelo os campos relevantes para a tarefa**.

O core diferencial de usar Pydantic aqui é:

**SEM PYDANTIC:** 

```python
call = response.output[0]

args = json.loads(call.arguments)
results = search_pubmed(**args)
```


**COM PYDANTIC:** 

```python
call = response.output[0]

args = model_validate_json(call.arguments)
results = search_pubmed(**args.model_dump())
```

Construção do **executor**- responsável por chamar o modelo, validar os argumentos e executar a função referente à tool:

In [ ]:
def make_call(call):
    if call.name != "search_pubmed":
        raise ValueError(f"Ferramenta desconhecida: {call.name}")

    # Validate the arguments and create an instance
    try:
        args = PubMedSearchArgs.model_validate_json(call.arguments)

    except ValidationError as error:
        result = {
            "status": "invalid_arguments",
            "errors": [
                {
                    "field": list(item["loc"]),
                    "message": item["msg"],
                }
                for item in error.errors()
            ],
        }

        return {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        }

    # Execute the existing function
    raw_results = search_pubmed(**args.model_dump())

    # Select the useful data to the agent
    esearch = raw_results["esearchresult"]

    result = {
        "status": "success",
        "source": "pubmed",
        "query": args.query,
        "requested_count": args.limit,
        "total_results": int(esearch["count"]),
        "returned_count": len(esearch["idlist"]),
        "pmids": esearch["idlist"],
    }

    # Return the observation to build the context and append it to the history
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    }

## **Agentic loop**

In [25]:
instructions = f"""
Você auxilia consultas bibliográficas no PubMed.

A query atual do projeto é:
{PUBMED_QUERY}

Orientações:
- Use como limit a quantidade explicitamente solicitada pelo usuário.
- Quando nenhuma quantidade for indicada, use {PUBMED_SAMPLE_SIZE}.
- Respeite os limites técnicos descritos no schema da ferramenta.
- Diferencie quantidade solicitada, total encontrado e quantidade retornada.
- Se a quantidade solicitada superar o total encontrado, explique que
  a consulta encontrou menos registros do que o solicitado.
- Baseie os números e PMIDs exclusivamente no resultado da ferramenta.
- Ao receber status invalid_arguments, examine os erros e corrija os
  argumentos antes de tentar novamente, respeitando o orçamento de chamadas.
"""

In [26]:
MAX_TOOL_CALLS = 2


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=[search_pubmed_tool],
        )

        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result_summary': call_output['output'][:200],
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

In [27]:
result = run_agent(
    "Quantos registros correspondem à query atual "
    "e retorne 120 pmids?"
)

result

iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":120}
iteration #1...
ASSISTANT:
A query encontrou **100 registros** no total.

Você solicitou **120 PMIDs**, mas a busca retornou **100 PMIDs**, porque há menos registros do que o solicitado.

PMIDs retornados:
41328123, 41076314, 40725551, 40393895, 40335417, 40153056, 40090550, 39756382, 39490003, 39487073, 39344460, 39344453, 39299812, 39256068, 39232864, 38743036, 38685147, 38071499, 37847424, 37246472, 36602552, 36540452, 36494246, 36201690, 35806971, 35727653, 35245490, 34809840, 34802615, 34503888, 34238634, 34112570, 33552802, 32997445

{'answer': 'A query encontrou **100 registros** no total.\n\nVocê solicitou **120 PMIDs**, mas a busca retornou **100 PMIDs**, porque há menos registros do que o solicitado.\n\nPMIDs retornados:\n41328123, 41076314, 40725551, 40393895, 40335417, 40153056, 40090550, 39756382, 39490003, 39487073, 39344460, 39344453, 39299812, 39256068, 39232864, 38743036, 38685147, 38071499, 37847424, 37246472, 36602552, 36540452, 36494246, 36201690, 35806971, 35727653, 35245490, 34809840, 34802615, 34503888, 34238634, 34112570, 33552802, 32997445, 32977036, 32917485, 32887620, 32866483, 32610046, 32569038, 32439381, 32328836, 32285518, 32035835, 31918888, 31594444, 31168422, 30965154, 30641280, 30615847, 30602388, 30316660, 29178982, 29077682, 28843402, 28711310, 28688869, 28687468, 28623686, 28412789, 28337986, 28161135, 29563484, 27717819, 27726907, 27487617, 27481673, 27134154, 26897300, 26889353, 26687154, 26611373, 25818493, 25483447, 25483442, 25102890, 25048029, 24861472, 24480620, 24035523, 2372

## **Using validation errors to the agent decide the next step**

Primeiro vamos simular o modelo cometendo um erro:

In [31]:
history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": "Consulte a query atual e mostre os primeiros 5 PMIDS."}   
]

A pergunta não exige que o agente explique as limitações da ferramenta, já que não é requerido `limit=0`, mas note que abaixo simulo o agente como estabelecendo que `limit=0`:

In [28]:
from types import SimpleNamespace

bad_call = SimpleNamespace(
    name="search_pubmed",
    call_id="call_validation_test",
    arguments=json.dumps({"query": PUBMED_QUERY, "limit": 0}),
)

Note que `bad_call` simula um `function_call` de `response.output`, ou seja, uma instância de `ResponseFunctionToolCall` com atributos `name`, `call_id` e `arguments`:

In [29]:
bad_call

namespace(name='search_pubmed',
          call_id='call_validation_test',
          arguments='{"query": "( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )", "limit": 0}')

Fazemos a construção de contexto levando em consideração que `bad_call` não é um objeto do SDK, então precisamos preencher os campos esperado pela API explicitamente, em vez de só contatenar `history += bad_call`:

In [ ]:
# Registra a tentativa de chamada
history.append({
    "type": "function_call",
    "name": bad_call.name,
    "call_id": bad_call.call_id,
    "arguments": bad_call.arguments,
})

# Registra a observação produzida pelo executor: a tool tipada!
history.append(make_call(bad_call))

history

[{'role': 'developer',
  'content': '\nVocê auxilia consultas bibliográficas no PubMed.\n\nA query atual do projeto é:\n( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/01"[Date - Publication] : "2025/12/31"[Date - Publication] )\n\nOrientações:\n- Use como limit a quantidade explicitamente solicitada pelo usuário.\n- Quando nenhuma quantidade for indicada, use 20.\n- Respeite os limites técnicos descritos no schema da ferramenta.\n- Diferencie quantidade solicitada, total encontrado e quantidade retornada.\n- Se a quantidade solicitada superar o total encontrado, explique que\n  a consulta encontrou menos registros do que o solicitado.\n- Baseie os números e PMIDs exclusivamente no resultado da ferramenta.\n- Ao receber st

O histórico contém, na ordem: **instruções**, **pergunta**, **tentativa inválida** e **observação com a rejeição da tool tipada**.

A seguir, fazemos uma próxima chamada ao modelo e ele deve corrigir seu raciocínio com base na observação gerada:

In [34]:
response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool],
    parallel_tool_calls=False,
)

for item in response.output:
    print(item.type, item)

function_call ResponseFunctionToolCall(arguments='{"query":"( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )","limit":5}', call_id='call_9HaHQ1n285d9c0nLAXXOQ9qN', name='search_pubmed', type='function_call', id='fc_0db3c9a0b7bb419f006ad39bfa06c087d28bd9439d906a2329', async_=None, caller=None, namespace=None, status='completed')


Conferimos agora a correção:

In [37]:
calls = [
    item for item in response.output
    if item.type == "function_call"
]

assert len(calls) == 1, "Confira a resposta: esperávamos uma nova chamada."

call = calls[0]

corrected_args = PubMedSearchArgs.model_validate_json(call.arguments)
print(corrected_args)

history.extend(response.output)
history.append(make_call(call))

final_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(final_response.output_text)

query='( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/01"[Date - Publication] : "2025/12/31"[Date - Publication] )' limit=5
Consulta realizada com sucesso.

- Total encontrado: 100
- Quantidade solicitada: 5
- Quantidade retornada: 5

Primeiros 5 PMIDs:
1. 41328123
2. 41076314
3. 40725551
4. 40393895
5. 40335417


O executor informa o erro, o modelo gera argumentos corrigidos, a ferramenta executa e o modelo responde com base no resultado

## **Agent with several tools**

Agora vamos repetir os procedimentos anteriores e construir um agente com três tools tipadas:

- `search_pubmed`: retorna pmids;
- `get_pubmed_citations`: retorna os citations a partir dos pmids;
- `get_crossref_work`: consulta DOI e obtem metadados complementares.

In [38]:
import httpx

from typing import Annotated
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from bibliometric_rag_assistant.ingestion.pubmed import (
    search_pubmed,
    fetch_pubmed_citations,
    parse_pubmed_xml,
)
from bibliometric_rag_assistant.ingestion.crossref import (
    fetch_crossref_work,
    parse_crossref_work,
)

Construção dos contratos usando Pydantic:

In [55]:
Pmid = Annotated[str, Field(pattern=r"^[0-9]+$")]

class PubMedSearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    query: str = Field(
        min_length=1,
        description=(
            "PubMed query using Boolean operators and field tags, "
            "such as [Title/Abstract] and [MeSH Terms]."
        ),
    )

    limit: int = Field(
        ge=1,
        le=10_000,
        description="Número máximo de PMIDs requisitados para esta busca. ",
    )


class PubMedCitationsArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    pmids: list[Pmid] = Field(
        min_length=1,
        description="PMIDs fornecidos pelo usuário ou retornados pela busca.",
    )


class CrossrefWorkArgs(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        strict=True,
        str_strip_whitespace=True,
    )

    doi: str = Field(
        min_length=1,
        description=(
            "DOI fornecido pelo usuário ou obtido nos metadados. "
            "Informe somente o DOI, sem o prefixo https://doi.org/."
        ),
    )

Funções que produzem as **observações entregues ao agente**:

In [62]:
def run_pubmed_search(query: str, limit: int) -> dict:
    raw_results = search_pubmed(query=query, limit=limit)
    esearch = raw_results["esearchresult"]

    return {
        "status": "success",
        "source": "pubmed",
        "query": query,
        "requested_count": limit,
        "total_results": int(esearch["count"]),
        "returned_count": len(esearch["idlist"]),
        "pmids": esearch["idlist"],
    }


def get_pubmed_citations(pmids: list[str]) -> dict:
    xml = fetch_pubmed_citations(pmids)
    citations = parse_pubmed_xml(xml)

    returned_pmids = {record["pmid"] for record in citations}

    return {
        "status": "success",
        "source": "pubmed",
        "requested_pmids": pmids,
        "returned_count": len(citations),
        "missing_pmids": [
            pmid for pmid in pmids
            if pmid not in returned_pmids
        ],
        "citations": citations,
    }


def get_crossref_work(doi: str) -> dict:
    try:
        payload = fetch_crossref_work(doi)

    except httpx.HTTPStatusError as error:
        if error.response.status_code != 404:
            raise

        return {
            "status": "not_found",
            "source": "crossref",
            "doi": doi,
        }

    return {
        "status": "success",
        "source": "crossref",
        "work": parse_crossref_work(payload["message"]),
    }

Gerando a descrição dos contratos:

In [63]:
TOOL_REGISTRY = {
    "search_pubmed": {
        "args_model": PubMedSearchArgs,
        "handler": run_pubmed_search,
        "description": search_pubmed_tool["description"],
    },
    "get_pubmed_citations": {
        "args_model": PubMedCitationsArgs,
        "handler": get_pubmed_citations,
        "description": (
            "Consulte metadados de PMIDs conhecidos no PubMed. "
            "Retorna título, DOI, periódico e ano de publicação. "
            "Use para obter detalhes dos registros encontrados na busca."
        ),
    },
    "get_crossref_work": {
        "args_model": CrossrefWorkArgs,
        "handler": get_crossref_work,
        "description": (
            "Consulte um DOI no Crossref. Retorna metadados e as contagens "
            "de referências e citações disponíveis nessa fonte. "
            "Requer um DOI fornecido pelo usuário ou obtido de outra ferramenta."
        ),
    },
}

TOOLS = [
    {
        "type": "function",
        "name": name,
        "description": spec["description"],
        "parameters": spec["args_model"].model_json_schema(),
        "strict": True,
    }
    for name, spec in TOOL_REGISTRY.items()
]

Ajustando o executor:

In [64]:
def make_call(call):
    if call.name not in TOOL_REGISTRY:
        raise ValueError(f"Ferramenta desconhecida: {call.name}")

    spec = TOOL_REGISTRY[call.name]

    try:
        args = spec["args_model"].model_validate_json(call.arguments)

    except ValidationError as error:
        result = {
            "status": "invalid_arguments",
            "errors": [
                {
                    "field": list(item["loc"]),
                    "message": item["msg"],
                }
                for item in error.errors()
            ],
        }

    else:
        try:
            result = spec["handler"](**args.model_dump())

        except httpx.HTTPStatusError as error:
            result = {
                "status": "external_error",
                "tool": call.name,
                "http_status": error.response.status_code,
                "message": "O serviço retornou um erro HTTP.",
            }

        except httpx.RequestError:
            result = {
                "status": "external_error",
                "tool": call.name,
                "message": "Não foi possível concluir a conexão com o serviço.",
            }

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    }

Ajustando as instruções do agente e o agentic loop:

In [65]:
instructions = f"""
Você auxilia consultas bibliográficas no PubMed.

A query atual do projeto é:
{PUBMED_QUERY}

Orientações:
- Use como limit a quantidade explicitamente solicitada pelo usuário.
- Quando nenhuma quantidade for indicada, use {PUBMED_SAMPLE_SIZE}.
- Respeite os limites técnicos descritos no schema da ferramenta.
- Diferencie quantidade solicitada, total encontrado e quantidade retornada.
- Se a quantidade solicitada superar o total encontrado, explique que
  a consulta encontrou menos registros do que o solicitado.
- Baseie os números e PMIDs exclusivamente no resultado da ferramenta.
- Ao receber status invalid_arguments, examine os erros e corrija os
  argumentos antes de tentar novamente, respeitando o orçamento de chamadas.
- Para obter detalhes de um PMID, use get_pubmed_citations.
- Para consultar o Crossref, utilize um DOI fornecido pelo usuário
  ou retornado por uma ferramenta. Nunca invente um DOI.
- Quando o DOI estiver ausente ou o Crossref retornar not_found,
  explique essa ausência.
- Diferencie contagem de referências de contagem de citações e
  identifique o Crossref como fonte dessas contagens.
- Se houver external_error, explique que a consulta não foi concluída.
"""

In [ ]:
MAX_TOOL_CALLS = 4


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=TOOLS,
            parallel_tool_calls=False,
        )
        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result': json.loads(call_output["output"]),
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

In [67]:
result = run_agent(
    "Use a query atual para obter o primeiro PMID. "
    "Consulte o título e o DOI desse registro no PubMed. "
    "Se houver DOI, consulte no Crossref as contagens de "
    "referências e citações. Informe as fontes e os dados ausentes."
)

iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":1}
iteration #1...
function_call: get_pubmed_citations {"pmids":["41328123"]}
iteration #2...
function_call: get_crossref_work {"doi":"10.7759/cureus.95806"}
iteration #3...
ASSISTANT:
Encontrei o primeiro registro da sua query e consultei seus metadados no PubMed e no Crossref.

### Registro encontrado
- **PMID:** 41328123
- **Título (PubMed):** *Evolving Horizons in Temporomandibular Joint Total Replacement: A Comprehensive Narrative Review.*
- **DOI (PubMed):** 10.7759/cureus.95806

### Crossref
- **Fonte:** Crossref
- **Título:** *Evolving

## **Evaluation of the agent with several tools**